# Code For Metric Calculation

In [ ]:
import asyncio
import httpx
import math
import json
import pandas as pd
import os
from datetime import datetime
from collections import defaultdict
from rapidfuzz import process, fuzz
from functools import lru_cache

# Handle Colab/Jupyter event loop issues
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

# -----------------------------
# CONFIGURATION
# -----------------------------
OPENALEX_API = "https://api.openalex.org"
HEADERS = {"User-Agent": "R2-Metric/1.0"}
CURRENT_YEAR = datetime.now().year
PER_PAGE = 200

# Ensure these files are in your directory
MEDIAN_CITATION_FILE = "/content/robust_subfield_medians2.txt"
MEDIAN_VENUE_FILE = "/content/merged_subfields_Vr.txt"
MEDIAN_COCITATION_FILE = "/content/global_max_co_r_results.jsonl"
SJR_CSV_PATH = "/content/all_journals_sjr_unique.csv"

WEIGHTS = {'citation': 0.5, 'venue': 0.35, 'cocitation': 0.15}

# -----------------------------
# CACHED & HELPER FUNCTIONS
# -----------------------------

@lru_cache(maxsize=10000)
def get_fuzzy_venue_match(title_l, candidates_tuple):
    match = process.extractOne(title_l, candidates_tuple, scorer=fuzz.token_sort_ratio, score_cutoff=90)
    return match[0] if match else None

def normalize_subfield_id(subfield_id):
    if not subfield_id: return ""
    return str(subfield_id).rsplit("/", 1)[-1].replace("S", "")

def get_primary_field(topics):
    if not topics: return "Unknown"
    sorted_topics = sorted(topics, key=lambda x: x.get("score", 0), reverse=True)
    primary_topic = sorted_topics[0]
    if "field" in primary_topic and "display_name" in primary_topic["field"]:
        return primary_topic["field"]["display_name"]
    return "Other"

# -----------------------------
# DATA LOADING
# -----------------------------

def load_all_medians():
    def load_file(path, key):
        d = {}
        if not os.path.exists(path):
            print(f"⚠️ Warning: Median file {path} not found.")
            return d
        try:
            with open(path, 'r') as f:
                for line in f:
                    if not line.strip(): continue
                    data = json.loads(line)
                    sid = normalize_subfield_id(data.get("subfield_id") or data.get("id") or "")
                    val = data.get(key, 1.0)
                    if sid: d[sid] = float(val) if float(val or 0) != 0 else 1.0
        except Exception as e: print(f"Error loading {path}: {e}")
        return d

    return {
        'cit': load_file(MEDIAN_CITATION_FILE, "median_score"),
        'ven': load_file(MEDIAN_VENUE_FILE, "median_Vr"),
        'coc': load_file(MEDIAN_COCITATION_FILE, "max_co_r")
    }

def load_sjr_quick(path):
    if not os.path.exists(path):
        print(f"⚠️ Warning: SJR file {path} not found.")
        return {}, {}, {}
    try:
        df = pd.read_csv(path)
        issn_map, title_map, fuzzy_idx = {}, {}, defaultdict(list)
        for _, row in df.iterrows():
            sjr = float(str(row.get("sjr") or row.get("SJR") or 0).replace(",", "."))
            t = str(row.get("title") or row.get("Title") or "").strip().lower()
            if t:
                title_map[t] = sjr
                fuzzy_idx[t[0]].append(t)
            issn = str(row.get("issn") or row.get("ISSN") or "")
            for i in issn.split(","):
                k = i.replace("-", "").strip()
                if k: issn_map[k] = sjr
        return issn_map, title_map, {k: tuple(v) for k, v in fuzzy_idx.items()}
    except Exception as e:
        print(f"SJR Load Error: {e}")
        return {}, {}, {}

# -----------------------------
# ASYNC CALCULATION LOGIC
# -----------------------------

async def calculate_researcher_metrics(client, name, medians, sjr_data):
    """Calculate R2 and metrics for a single researcher name"""
    print(f"🔍 Searching for: {name}...")

    try:
        # 1. Fetch Author info
        r = await client.get(f"{OPENALEX_API}/authors", params={"search": name, "per-page": 1})
        res = r.json().get("results")
        if not res:
            return {"status": "Error", "message": f"Researcher '{name}' not found on OpenAlex."}

        author = res[0]
        auth_id = author['id'].split("/")[-1] if author.get('id') else None

        if not auth_id:
            return {"status": "Error", "message": "No valid author ID found."}

        # Get detailed author info
        author_detail_r = await client.get(f"{OPENALEX_API}/authors/{auth_id}")
        author_detail = author_detail_r.json()

        h_index = author_detail.get('summary_stats', {}).get('h_index', 0)
        total_papers = author_detail.get('works_count', 0)
        cited_by_count = author_detail.get('cited_by_count', 0)

        # 2. Fetch Works
        works = []
        cursor = "*"
        while cursor:
            w_r = await client.get(f"{OPENALEX_API}/works", params={
                "filter": f"author.id:{auth_id}",
                "cursor": cursor,
                "per-page": PER_PAGE,
                "select": "id,cited_by_count,publication_year,authorships,topics,primary_location,referenced_works"
            })
            w_data = w_r.json()
            works.extend(w_data.get("results", []))
            cursor = w_data["meta"].get("next_cursor")
            if not w_data.get("results"): break

        if not works:
            return {"status": "Error", "message": "No works found for this author."}

        # 3. Process R2
        auth_work_ids = {w['id'] for w in works}
        sf_stats = defaultdict(lambda: {'cit_score': 0.0, 'ven_num': 0.0, 'ven_den': 0.0, 'coc_score': 0.0, 'count': 0})
        all_topics = []

        for w in works:
            if w.get("topics"): all_topics.extend(w["topics"])

            topics = sorted(w.get("topics", []), key=lambda x: x.get("score", 0), reverse=True)
            sf_id = next((normalize_subfield_id(t["subfield"]["id"]) for t in topics if t.get("subfield")), None)
            if not sf_id: continue

            # Position
            idx, p_r = 1, 1
            authorships = w.get("authorships", [])
            for i, a in enumerate(authorships, start=1):
                author_id = a.get("author", {}).get("id", "")
                if isinstance(author_id, str) and author_id.endswith(auth_id):
                    idx = i
                    pos = a.get("author_position")
                    p_r = 1 if pos == "first" else len(authorships) if pos == "last" else i
                    break

            cits = w.get("cited_by_count", 0)
            age = max(1, CURRENT_YEAR - (w.get("publication_year") or CURRENT_YEAR))
            cp = math.log1p((cits * (1.0/idx)) / math.sqrt(age))

            # SJR
            sjr = None
            loc = w.get("primary_location")
            if loc and loc.get("source"):
                src = loc["source"]
                for issn in (src.get("issn") or []):
                    k = issn.replace("-", "")
                    if k in sjr_data[0]: sjr = sjr_data[0][k]; break
                if sjr is None:
                    t = (src.get("display_name") or "").lower().strip()
                    if t in sjr_data[1]: sjr = sjr_data[1][t]

            # Cocitation
            f_p = sum(1 for ref in w.get("referenced_works", []) if ref in auth_work_ids)
            co_r = f_p / (p_r * math.sqrt(cits)) if (cits > 0 and f_p > 0) else 0

            s = sf_stats[sf_id]
            s['cit_score'] += cp
            s['coc_score'] += co_r
            s['count'] += 1
            if sjr is not None:
                s['ven_num'] += cp * sjr
                s['ven_den'] += cp

        # 4. Final Calculation
        total_r2 = 0
        total_calc_papers = len(works)
        for sid, s in sf_stats.items():
            v_raw = s['ven_num'] / s['ven_den'] if s['ven_den'] > 0 else 0
            n_cit = s['cit_score'] / medians['cit'].get(sid, 1.0)
            n_ven = v_raw / medians['ven'].get(sid, 1.0)
            n_coc = s['coc_score'] / medians['coc'].get(sid, 1.0)
            r2_sub = (WEIGHTS['citation'] * n_cit) + (WEIGHTS['venue'] * n_ven) + (WEIGHTS['cocitation'] * n_coc)
            total_r2 += (r2_sub * (s['count'] / total_calc_papers))

        return {
            'status': 'Success',
            'full_name': author['display_name'],
            'field': get_primary_field(all_topics),
            'r2': round(total_r2, 4),
            'h_index': h_index,
            'total_papers': total_papers,
            'total_citations': cited_by_count,
            'openalex_id': auth_id
        }

    except Exception as e:
        import traceback
        print(f"Error details: {traceback.format_exc()}")
        return {"status": "Error", "message": str(e)}

# -----------------------------
# MAIN EXECUTION
# -----------------------------

async def main():
    print("--- Researcher R2 Calculator ---")

    # 1. Get input name
    target_name = input("Enter the researcher's name: ").strip()
    if not target_name:
        print("No name entered. Exiting.")
        return

    # 2. Load supporting data
    print("⏳ Loading median and SJR data (this may take a moment)...")
    medians = load_all_medians()
    sjr_data = load_sjr_quick(SJR_CSV_PATH)

    # 3. Execute
    async with httpx.AsyncClient(headers=HEADERS, timeout=60.0) as client:
        result = await calculate_researcher_metrics(client, target_name, medians, sjr_data)

    # 4. Display Results
    print("\n" + "="*40)
    if result['status'] == 'Success':
        print(f"RESULTS FOR: {result['full_name']}")
        print(f"Primary Field: {result['field']}")
        print("-" * 20)
        print(f"⭐ R2 Score:      {result['r2']}")
        print(f"📈 H-Index:       {result['h_index']}")
        print(f"📄 Total Papers:  {result['total_papers']}")
        print(f"💬 Total Cites:   {result['total_citations']}")
        print(f"🆔 OpenAlex ID:   {result['openalex_id']}")
    else:
        print(f"❌ Error: {result['message']}")
    print("="*40 + "\n")

if __name__ == "__main__":
    asyncio.run(main())